In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os
import scipy
import sys

sys.path.append('../Exp6 - iENG vs sENG')
import shared_iENG_sENG
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D


def X_y_from_matfile(path, modality, session):
    X, y = [], []

    if modality=='EMG':
        bluetooth_id = 'E8DD80E550BB'
    elif modality=='ENG':
        bluetooth_id = 'E9AD0E7DCC2B'

    data_per_class_files = os.listdir(path+f'{modality}_{session}/{bluetooth_id}/raw/')

    for cls in data_per_class_files:
        input_path = path+f'{modality}_{session}/{bluetooth_id}/raw/{cls}/'
        files = os.listdir(input_path)

        mat = scipy.io.loadmat(input_path+files[0])
        x_tmp = mat['Data_Fea'].transpose(2, 0, 1)

        X.append(x_tmp.reshape(x_tmp.shape[0], x_tmp.shape[1]*x_tmp.shape[2]))
        y.append(mat['Data_Cls'].ravel())

    return X, y

# invasive ENG (Amputee)

In [ ]:
path = 'C:/Users/hml76/Desktop/Jupyter/Federated Prototype Learning/Dataset/Ours_cleaned/'
fname = os.listdir(path)

iENG_subjects = {
    "subject2": {"X": [], "Y": []},
    "subject3": {"X": [], "Y": []},
}

for f in fname:
    for i in range(10):
        X = np.array(pd.read_csv(path + f'{f}/rep{i}_X.csv'))
        y = np.array(pd.read_csv(path + f'{f}/rep{i}_Y.csv'))
        #indices = np.random.permutation(len(X)) # shuffle
        #X, y = X[indices], y[indices]

        y_int = np.argmax(y, axis=1)
        num_classes = np.max(y_int) + 1

        if "subject2" in f:
            iENG_subjects["subject2"]["X"].append(X)
            iENG_subjects["subject2"]["Y"].append(y_int)
        elif "subject3" in f:
            iENG_subjects["subject3"]["X"].append(X)
            iENG_subjects["subject3"]["Y"].append(y_int)

In [ ]:
final_session = 5
y = iENG_subjects["subject2"]['Y'][2:final_session] + iENG_subjects["subject3"]['Y'][2:final_session]
x = iENG_subjects["subject2"]['X'][2:final_session] + iENG_subjects["subject3"]['X'][2:final_session]

y = np.concatenate(y, axis=0)
x = np.concatenate(x, axis=0)
x = x.reshape(x.shape[0], 16, 14)[:, [0, 4, 8, 12], :].reshape(x.shape[0], 4*14)   # shape (N, 4, 14)

mask = (y == 0)
iENG_x_zero = x[mask]
iENG_y_zero = y[mask]

iENG_x_zero.shape, iENG_y_zero.shape

# sENG (HM, BY, MJ)

In [ ]:
bluetooth_id = 'E9AD0E7DCC2B'
base_path = 'C:/Users/hml76/PycharmProjects/MindForce/data/sENG_iENG/'
X_sENG_sub1, y_sENG_sub1 = shared_iENG_sENG.return_X_y_get_from_matfile(path = base_path+f'Hunmin/{bluetooth_id}/raw/', num_session=[0,1,2,3,4], balance=True)
X_sENG_sub2, y_sENG_sub2 = shared_iENG_sENG.return_X_y_get_from_matfile(path = base_path+f'Byeongchan/{bluetooth_id}/raw/', num_session=[0,1,2,3,4], balance=True)
X_sENG_sub3, y_sENG_sub3 = shared_iENG_sENG.return_X_y_get_from_matfile(path = base_path+f'Minjeong/{bluetooth_id}/raw/', num_session=[0,1,2,3,4], balance=True)
X_sENG, y_sENG = np.concatenate([X_sENG_sub1, X_sENG_sub2, X_sENG_sub3], axis=0), np.concatenate([y_sENG_sub1, y_sENG_sub2, y_sENG_sub3], axis=0)

'''
y = y_sENG_sub1.tolist() + y_sENG_sub2.tolist() + y_sENG_sub3.tolist()
y = np.array(y)
#y = np.concatenate(y, axis=0)
x1 = X_sENG_sub1.reshape(X_sENG_sub1.shape[0], X_sENG_sub1.shape[1]*X_sENG_sub1.shape[2])#.tolist()
x2 = X_sENG_sub2.reshape(X_sENG_sub2.shape[0], X_sENG_sub2.shape[1]*X_sENG_sub2.shape[2])#.tolist()
x3 = X_sENG_sub3.reshape(X_sENG_sub3.shape[0], X_sENG_sub3.shape[1]*X_sENG_sub3.shape[2])#.tolist()
x = x1.tolist()+x2.tolist()+x3.tolist()
x = np.array(x)
'''

x, y = X_sENG, y_sENG

mask = (y == 0)
sENG_x_zero = x[mask]
sENG_y_zero = y[mask]

sENG_x_zero.shape, np.array(sENG_y_zero).shape

In [ ]:
base_path = 'C:/Users/hml76/PycharmProjects/MindForce/data/EMG_ENG/'
X_sENG_Sub1, y_sENG_Sub1 = X_y_from_matfile(path=base_path+f'Hunmin/', modality='ENG', session='v1')
X_sENG_Sub2, y_sENG_Sub2 = X_y_from_matfile(path=base_path+f'Carlson/', modality='ENG', session='v1')
X_sENG_Sub3, y_sENG_Sub3 = X_y_from_matfile(path=base_path+f'Jongin/', modality='ENG', session='v1')
#X_sENG_Sub4, y_sENG_Sub4 = X_y_from_matfile(path=base_path+f'Hunmin/', modality='ENG', session='v1')
#X_sENG_Sub5, y_sENG_Sub5 = X_y_from_matfile(path=base_path+f'Carlson/', modality='ENG', session='v1')
#X_sENG_Sub6, y_sENG_Sub6 = X_y_from_matfile(path=base_path+f'Jongin/', modality='ENG', session='v1')

#print(X_sEMG_Sub1.shape, X_sEMG_Sub2.shape, X_sEMG_Sub3.shape)
final_session = 2
y = y_sENG_Sub1[0:final_session] + y_sENG_Sub2[0:final_session] + y_sENG_Sub3[0:final_session]# + y_sENG_Sub4[0:final_session] + y_sENG_Sub5[0:final_session] + y_sENG_Sub6[0:final_session]
y = np.concatenate(y, axis=0)
x = X_sENG_Sub1[0:final_session] + X_sENG_Sub2[0:final_session] + X_sENG_Sub3[0:final_session]# + X_sENG_Sub4[0:final_session] + X_sENG_Sub5[0:final_session] + X_sENG_Sub6[0:final_session]
x = np.concatenate(x, axis=0)

mask = (y == 0)
sENG_x_zero = x[mask]
sENG_y_zero = y[mask]

sENG_x_zero.shape, sENG_y_zero.shape

# sEMG (HJ, YC, XY)

In [ ]:
base_path = 'C:/Users/hml76/PycharmProjects/MindForce/data/EMG_ENG/'
X_sEMG_Sub1, y_sEMG_Sub1 = X_y_from_matfile(path=base_path+f'Hongjun/', modality='EMG', session='v1')
#y_sEMG_Sub1 = X_y_from_matfile(path=base_path+f'{subject}/', modality='ENG', session='v1')
X_sEMG_Sub2, y_sEMG_Sub2 = X_y_from_matfile(path=base_path+f'Youngchul/', modality='EMG', session='v1')
X_sEMG_Sub3, y_sEMG_Sub3 = X_y_from_matfile(path=base_path+f'Xianyu/', modality='EMG', session='v1')
#X_sEMG_Sub4, y_sEMG_Sub4 = X_y_from_matfile(path=base_path+f'Hunmin/', modality='EMG', session='v1')
#X_sEMG_Sub5, y_sEMG_Sub5 = X_y_from_matfile(path=base_path+f'Carlson/', modality='EMG', session='v1')
#X_sEMG_Sub6, y_sEMG_Sub6 = X_y_from_matfile(path=base_path+f'Jongin/', modality='EMG', session='v1')

#print(X_sEMG_Sub1.shape, X_sEMG_Sub2.shape, X_sEMG_Sub3.shape)
final_session = 2
y = y_sEMG_Sub1[0:final_session] + y_sEMG_Sub2[0:final_session] + y_sEMG_Sub3[0:final_session]# + y_sEMG_Sub4[0:final_session] + y_sEMG_Sub5[0:final_session] + y_sEMG_Sub6[0:final_session]
y = np.concatenate(y, axis=0)
x = X_sEMG_Sub1[0:final_session] + X_sEMG_Sub2[0:final_session] + X_sEMG_Sub3[0:final_session]# + X_sEMG_Sub4[0:final_session] + X_sEMG_Sub5[0:final_session] + X_sEMG_Sub6[0:final_session]
x = np.concatenate(x, axis=0)

mask = (y == 0)
sEMG_x_zero = x[mask]
sEMG_y_zero = y[mask]

sEMG_x_zero.shape, sEMG_y_zero.shape

# T-SNE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from scipy.spatial.distance import cdist

def t_sne_compute_dist(sEMG, sENG, iENG):

    # Stack all for t-SNE
    X_all = np.vstack([sEMG, sENG, iENG])
    labels = np.array([0]*sEMG.shape[0] + [1]*sENG.shape[0] + [2]*iENG.shape[0])

    # t-SNE
    tsne = TSNE(n_components=2, random_state=42)
    X_tsne = tsne.fit_transform(X_all)

    # Visualize
    plt.figure(figsize=(8,6))
    colors = ['r', 'g', 'b']
    names = ['sEMG', 'sENG', 'iENG']

    for i, c, name in zip(range(3), colors, names):
        plt.scatter(X_tsne[labels==i, 0], X_tsne[labels==i, 1], c=c, label=name, alpha=0.6, s=20)

    plt.legend()
    plt.title("t-SNE of three datasets")
    plt.show()

    # Compute prototypes (mean feature vector)
    proto_i = sEMG.mean(axis=0)
    proto_s1 = sENG.mean(axis=0)
    proto_s2 = iENG.mean(axis=0)

    # Stack prototypes
    prototypes = np.vstack([proto_i, proto_s1, proto_s2])

    # Compute pairwise distances
    dist_matrix = cdist(prototypes, prototypes, metric='euclidean')
    dist_matrix = pd.DataFrame(dist_matrix, index=names, columns=names)
    #print("Pairwise distances between prototypes:\n", dist_matrix)
    return dist_matrix


In [ ]:
dist_all = []

for feat in range(14):
    print(sEMG_x_zero.shape, sENG_x_zero.shape, iENG_x_zero.shape)
    sEMG_x_zero = sEMG_x_zero.reshape(sEMG_x_zero.shape[0], 4, 14)
    sEMG_x_one_feat = sEMG_x_zero[:, :, feat:feat+1].reshape(sEMG_x_zero.shape[0], 4)

    sENG_x_zero = sENG_x_zero.reshape(sENG_x_zero.shape[0], 4, 14)
    sENG_x_one_feat = sENG_x_zero[:, :, feat:feat+1].reshape(sENG_x_zero.shape[0], 4)

    iENG_x_zero = iENG_x_zero.reshape(iENG_x_zero.shape[0], 4, 14)
    iENG_x_one_feat = iENG_x_zero[:, :, feat:feat+1].reshape(iENG_x_zero.shape[0], 4)

    print(sEMG_x_one_feat.shape, sENG_x_one_feat.shape, iENG_x_one_feat.shape)
    dist = t_sne_compute_dist(sEMG_x_one_feat, sENG_x_one_feat, iENG_x_one_feat)

    print(dist)
    dist_all.append(dist)

In [ ]:
#sEMG-sENG:
#sENG-iENG:
#sEMG-iENG:

sEMG_x_zero = sEMG_x_zero.reshape(sEMG_x_zero.shape[0], 4*14)
sENG_x_zero = sENG_x_zero.reshape(sENG_x_zero.shape[0], 4*14)
iENG_x_zero = iENG_x_zero.reshape(iENG_x_zero.shape[0], 4*14)
print(sEMG_x_zero.shape, sENG_x_zero.shape, iENG_x_zero.shape)

dist = t_sne_compute_dist(sEMG_x_zero, sENG_x_zero, iENG_x_zero)
dist

In [ ]:

#sEMG-sENG:
#sENG-iENG:
#sEMG-iENG:

sEMG_x_zero = sEMG_x_zero.reshape(sEMG_x_zero.shape[0], 4*14)
sENG_x_zero = sENG_x_zero.reshape(sENG_x_zero.shape[0], 4*14)
iENG_x_zero = iENG_x_zero.reshape(iENG_x_zero.shape[0], 4*14)
print(sEMG_x_zero.shape, sENG_x_zero.shape, iENG_x_zero.shape)

dist = t_sne_compute_dist(sEMG_x_zero, sENG_x_zero, iENG_x_zero)
dist

In [ ]:
#one subject - one session
#sEMG-sENG: 6.03
#sENG-iENG: 3.84
#sEMG-iENG: 5.48


dist_df = pd.DataFrame(np.mean(dist_all, axis=0), index=['sEMG', 'sENG', 'iENG'], columns=['sEMG', 'sENG', 'iENG'])
dist_df

In [ ]:
feature_names = ['Zero Crossing (ZC)', 'Slope Sign Changes (SSC)', 'Waveform Length (WL)', 'WAMP', 'Mean Absolute Value (MAV)', 'Mean Square (MS)',
                 'Root Mean Square (RMS)', 'v-order 3 (V3)', 'log detector (LD)', 'difference absolute standard deviation value (DASDV)', 'maximum fractal length (MFL)',
                 'myopulse percentage rate (MPR)', 'mean absolute value slope (MAVS)', 'weighted mean absolute (WMS)']

dist_df = pd.DataFrame(np.mean(dist_all, axis=1), index=feature_names, columns=['sEMG', 'sENG', 'iENG'])
dist_df

In [ ]:
dist_vals = []

for f in range(14):
    sEMG_sENG = float(dist_all[f][0][1])
    sEMG_iENG = float(dist_all[f][0][2])
    sENG_iENG = float(dist_all[f][1][2])
    dist_vals.append([sEMG_sENG, sEMG_iENG, sENG_iENG])

#sns.heatmap(dist_vals, cmap='YlGnBu', fmt=)
#plt.show()
pd.DataFrame(dist_vals, columns=['sEMG-sENG', 'sEMG-iENG', 'sENG-iENG'])

1. feature-dimension
2. gesture-dimension
3.